## processing the datasets for the ML model



this file is intended to process three datasets (kepler, tess, k2) respictivly and merge the three datasets with the most important features for the model.Unfortuntly the k2 dataset had too many missing values (~80%) in some features that were important , so we reside to use only two (kepler, tess).First we clean the data and handle missing values. Then we rename columns to unify the datasets,finally merging them. In the end the MERGED dataset will be saved and used in the next step which is training the model.
I Hope you enjoy ready through this mess :)

Load the Kepler dataset

In [19]:
import pandas as pd
# Downloaded CSV from NASA archive
koi_data = pd.read_csv("../datasets/cumulative.csv")


# Keep only useful columns

print(koi_data.head())

   rowid     kepid kepoi_name   kepler_name koi_disposition koi_pdisposition  \
0      1  10797460  K00752.01  Kepler-227 b       CONFIRMED        CANDIDATE   
1      2  10797460  K00752.02  Kepler-227 c       CONFIRMED        CANDIDATE   
2      3  10811496  K00753.01           NaN  FALSE POSITIVE   FALSE POSITIVE   
3      4  10848459  K00754.01           NaN  FALSE POSITIVE   FALSE POSITIVE   
4      5  10854555  K00755.01  Kepler-664 b       CONFIRMED        CANDIDATE   

   koi_score  koi_fpflag_nt  koi_fpflag_ss  koi_fpflag_co  ...  \
0      1.000              0              0              0  ...   
1      0.969              0              0              0  ...   
2      0.000              0              1              0  ...   
3      0.000              0              1              0  ...   
4      1.000              0              0              0  ...   

   koi_steff_err2  koi_slogg  koi_slogg_err1  koi_slogg_err2  koi_srad  \
0           -81.0      4.467           0.064    

see more info

In [20]:
koi_data.describe()

,rowid,kepid,koi_score,koi_fpflag_nt,koi_fpflag_ss,koi_fpflag_co,koi_fpflag_ec,koi_period,koi_period_err1,koi_period_err2,...,koi_steff_err2,koi_slogg,koi_slogg_err1,koi_slogg_err2,koi_srad,koi_srad_err1,koi_srad_err2,ra,dec,koi_kepmag
count,9564.000000,9.564000e+03,8054.000000,9564.000000,9564.000000,9564.000000,9564.000000,9564.000000,9110.000000,9110.000000,...,9081.000000,9201.000000,9096.000000,9096.000000,9201.000000,9096.000000,9096.000000,9564.000000,9564.000000,9563.000000
mean,4782.500000,7.690628e+06,0.480829,0.188206,0.231598,0.194898,0.120033,75.671358,0.002148,-0.002148,...,-162.265059,4.310157,0.120738,-0.143161,1.728712,0.362292,-0.394806,292.060163,43.810433,14.264606
std,2761.033321,2.653459e+06,0.476928,0.390897,0.421875,0.396143,0.325018,1334.744046,0.008236,0.008236,...,72.746348,0.432606,0.132837,0.085477,6.127185,0.930870,2.168213,4.766657,3.601243,1.385448
min,1.000000,7.574500e+05,0.000000,0.000000,0.000000,0.000000,0.000000,0.241843,0.000000,-0.172500,...,-1762.000000,0.047000,0.000000,-1.207000,0.109000,0.000000,-116.137000,279.852720,36.577381,6.966000
25%,2391.750000,5.556034e+06,0.000000,0.000000,0.000000,0.000000,0.000000,2.733684,0.000005,-0.000276,...,-198.000000,4.218000,0.042000,-0.196000,0.829000,0.129000,-0.250000,288.660770,40.777173,13.440000
50%,4782.500000,7.906892e+06,0.334000,0.000000,0.000000,0.000000,0.000000,9.752831,0.000035,-0.000035,...,-160.000000,4.438000,0.070000,-0.128000,1.000000,0.251000,-0.111000,292.261125,43.677504,14.520000
75%,7173.250000,9.873066e+06,0.998000,0.000000,0.000000,0.000000,0.000000,40.715178,0.000276,-0.000005,...,-114.000000,4.543000,0.149000,-0.088000,1.345000,0.364000,-0.069000,295.859160,46.714611,15.322000
max,9564.000000,1.293514e+07,1.000000,1.000000,1.000000,1.000000,1.000000,129995.778400,0.172500,0.000000,...,0.000000,5.364000,1.472000,0.000000,229.908000,33.091000,0.000000,301.720760,52.336010,20.003000


Removing columns: Six columns were selectively removed due to their lack of substantive
contribution to the predictive modeling. These columns were found to serve
solely as identifiers, devoid of significant relevance to the prediction task at hand.

In [21]:
koi_data =koi_data.drop(["rowid", "kepid", "kepoi_name", "kepler_name","koi_pdisposition", "koi_score"], axis=1)

The following columns “koi_teq_err1” and “koi_teq_err2” were also removed because
they were completely empty.

In [22]:
koi_data = koi_data.drop(["koi_teq_err1", "koi_teq_err2"], axis=1)

Here we define the important feature that will be used and drop the rest

In [186]:
koi_selected_feat=["koi_period",
    "koi_time0bk",
    "koi_duration",
    "koi_depth",
    "koi_prad",
    "koi_insol",
    "koi_teq",
    "koi_steff",
    "koi_slogg",
    "koi_srad",
    "ra",
    "dec",
    "koi_kepmag"]
#for the one row missing in koi_meg
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="median")
koi_data[koi_selected_feat] = imputer.fit_transform(koi_data[koi_selected_feat])



In [ ]:
#adding missing flags
for col in koi_data.columns:
    if koi_data[col].isna().sum() > 0:
        koi_data[col + "_missing"] = koi_data[col].isna().astype(int)


In [187]:
koi_data.isnull().sum()

koi_disposition               0
koi_fpflag_nt                 0
koi_fpflag_ss                 0
koi_fpflag_co                 0
koi_fpflag_ec                 0
koi_period                    0
koi_time0bk                   0
koi_impact                  363
koi_duration                  0
koi_depth                     0
koi_prad                      0
koi_teq                       0
koi_insol                     0
koi_model_snr               363
koi_tce_plnt_num            346
koi_steff                     0
koi_slogg                     0
koi_srad                      0
ra                            0
dec                           0
koi_kepmag                    0
koi_impact_missing            0
koi_depth_missing             0
koi_prad_missing              0
koi_teq_missing               0
koi_insol_missing             0
koi_model_snr_missing         0
koi_tce_plnt_num_missing      0
koi_steff_missing             0
koi_slogg_missing             0
koi_srad_missing              0
koi_kepm

In [189]:

import json

with open("../datasets/KOI_selected_features.json", "w") as f:
    json.dump({"features":koi_selected_feat, "target":"koi_disposition"}, f)



now we save the important features in a new csv file

In [190]:
koi_data= koi_data[koi_selected_feat + ["koi_disposition"]]
koi_data.to_csv("../datasets/cleaned_koi_data.csv")


encoding 1 = confirmed , 2= false positive, 0 candidte

pipline definition

## THis  for the TOI dataset


In [150]:
TOI_data = pd.read_csv("../datasets/TOI_2024.07.03_19.07.30.csv", comment ="#")
TOI_data.shape

(7203, 65)

First lets extract our important features

In [151]:
selected_features =['pl_orbper', 'pl_trandurh', 'pl_trandep', 'pl_rade', 'pl_insol',
 'pl_eqt', 'st_teff', 'st_logg', 'st_rad', 'st_tmag', 'ra', 'dec']
target="tfopwg_disp"
TOI_data = TOI_data[selected_features+ [target]]

In [152]:
TOI_data[target].isnull().sum()

np.int64(2)

In [154]:
from sklearn.impute import SimpleImputer

num_imputer = SimpleImputer(strategy="median")
TOI_data[selected_features]= num_imputer.fit_transform(TOI_data[selected_features])

TOI_data= TOI_data.dropna(subset=target)
print("✅ Missing values handled successfully!")
print(TOI_data[selected_features].head)






✅ Missing values handled successfully!
<bound method NDFrame.head of       pl_orbper  pl_trandurh    pl_trandep    pl_rade      pl_insol  \
0      2.171348     2.017220    656.886099   5.818163  22601.948581   
1      1.931646     3.166000   1286.000000  11.215400  44464.500000   
2      1.867557     1.408000   1500.000000  23.752900   2860.610000   
3      2.743230     3.167000    383.410000  10.509800   1177.360000   
4      3.573014     3.370000    755.000000  11.311300  54679.300000   
...         ...          ...           ...        ...           ...   
7198   3.443800     2.572000   7260.750000  10.509800   1413.670000   
7199  14.537800     6.826000   4040.000000  16.052300    365.719000   
7200   8.413486     3.556833  17479.605331  20.510696    127.916421   
7201   0.941436     1.360700    339.912662   4.529209  41562.587811   
7202   1.507896     2.759000   1839.000000   5.530910   9662.800000   

           pl_eqt  st_teff  st_logg    st_rad    st_tmag          ra  \
0     

In [157]:
TOI_data[selected_features].isnull().sum()


pl_orbper      0
pl_trandurh    0
pl_trandep     0
pl_rade        0
pl_insol       0
pl_eqt         0
st_teff        0
st_logg        0
st_rad         0
st_tmag        0
ra             0
dec            0
dtype: int64

In [158]:
TOI_data.to_csv("../datasets/cleaned_TOI_data.csv")

In [159]:
import json

# Save
with open("../datasets/TOI_features.json", "w") as f:
    json.dump({"features": selected_features, "target": target}, f)

print("✅ Features and target saved to features.json")


✅ Features and target saved to features.json


### cleaning the k2 dataset

In [115]:
k2_data = pd.read_csv("../datasets/k2_dataset.csv", comment="#")
k2_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4004 entries, 0 to 4003
Columns: 295 entries, rowid to pl_ndispec
dtypes: float64(260), int64(2), object(33)
memory usage: 9.0+ MB


The only categorical column we need is the disposition

In [116]:
cat_cols = k2_data.select_dtypes(include=["object", "category"]).columns
keep_col ="disposition"
k2_data = k2_data.drop(columns=[col for col in cat_cols if col != keep_col])

THen we need to define the features we want because they are alot

In [117]:
k2_features = [
    "pl_orbper",     # Orbital period [days]
    "pl_tranmid",    # Transit midpoint [days]
    "pl_trandur",    # Transit duration [hours]
    "pl_trandep",    # Transit depth [%]
    "pl_rade",       # Planet radius [Earth radii]
    "pl_insol",      # Insolation flux [Earth flux]
    "pl_eqt",        # Equilibrium temperature [K]
    "st_teff",       # Stellar effective temperature [K]
    "st_logg",       # Stellar log(g) [cm/s^2]
    "st_rad",        # Stellar radius [Solar radius]
    "ra",            # Right ascension [deg]
    "dec",           # Declination [deg]
    "sy_tmag"        # TESS magnitude
]
k2_data = k2_data[k2_features+[keep_col]]

Now we can handle the missing values of our reduced dataset

In [118]:
k2_data.isnull().sum()

pl_orbper        67
pl_tranmid       88
pl_trandur     1237
pl_trandep     1919
pl_rade         845
pl_insol       3375
pl_eqt         3159
st_teff        1127
st_logg        1657
st_rad          148
ra               23
dec              23
sy_tmag          27
disposition       0
dtype: int64

clearly we have too much missing important data so we're aborting thes step for now and train only on the kepler and tess data


## Here we merge the three datasets (kepler , tess ) for better genarisation and to predict from any dataset

In [191]:
KOI_data = pd.read_csv('../datasets/cleaned_koi_data.csv')
TOI_data = pd.read_csv('../datasets/cleaned_TOI_data.csv')

In [192]:
mapping = {
    # KOI side : (common_name, TOI side)

    # Disposition / target label
    "koi_disposition": ("target", "tfopwg_disp"),

    # Orbital / transit geometry
    "koi_period": ("orbital_period", "pl_orbper"),
    "koi_time0bk": ("transit_midpoint", "pl_tranmid"),
    "koi_duration": ("transit_duration", "pl_trandurh"),
    "koi_depth": ("transit_depth", "pl_trandep"),

    # Planet physical properties
    "koi_prad": ("planet_radius", "pl_rade"),
    "koi_insol": ("insolation_flux", "pl_insol"),
    "koi_teq": ("equilibrium_temperature", "pl_eqt"),

    # Stellar properties (host star)
    "koi_steff": ("stellar_teff", "st_teff"),
    "koi_slogg": ("stellar_logg", "st_logg"),
    "koi_srad": ("stellar_radius", "st_rad"),

    # Photometric / magnitude
    "koi_kepmag": ("stellar_mag", "st_tmag"),

    # Position coordinates
    "ra": ("ra", "ra"),
    
    "dec": ("dec", "dec")
}

renameing the columns to tess names


In [210]:
rename_dict_tess = {k: v[0] for k, v in mapping.items()}
KOI_renamed = KOI_data.rename(columns=rename_dict_tess)

rename_dict_toi = {v[1]: v[0] for k, v in mapping.items() if v[1] in TOI_data.columns}  # TOI columns → unified names
TOI_renamed = TOI_data.rename(columns=rename_dict_toi)
print(KOI_renamed.head())


   Unnamed: 0  orbital_period  transit_midpoint  transit_duration  \
0           0        9.488036        170.538750           2.95750   
1           1       54.418383        162.513840           4.50700   
2           2       19.899140        175.850252           1.78220   
3           3        1.736952        170.307565           2.40641   
4           4        2.525592        171.595550           1.65450   

   transit_depth  planet_radius  insolation_flux  equilibrium_temperature  \
0          615.8           2.26            93.59                    793.0   
1          874.8           2.83             9.11                    443.0   
2        10829.0          14.60            39.30                    638.0   
3         8079.2          33.46           891.96                   1395.0   
4          603.3           2.75           926.16                   1406.0   

   stellar_teff  stellar_logg  stellar_radius         ra        dec  \
0        5455.0         4.467           0.927  291.

In [211]:
TOI_renamed.isnull().sum()

Unnamed: 0                 0
orbital_period             0
transit_duration           0
transit_depth              0
planet_radius              0
insolation_flux            0
equilibrium_temperature    0
stellar_teff               0
stellar_logg               0
stellar_radius             0
stellar_mag                0
ra                         0
dec                        0
target                     0
dtype: int64

In [212]:
#now we drop only extract the common important features
selected_merge_features = [
    "orbital_period", "transit_duration", "transit_depth", "planet_radius",
    "insolation_flux", "equilibrium_temperature", "stellar_teff",
    "stellar_logg", "stellar_radius", "stellar_mag", "ra", "dec"
]
target = "target"

KOI_data_selected = KOI_renamed[selected_merge_features + [target]]
TOI_data_selected = TOI_renamed[selected_merge_features + [target]]


Look at the target data of tess to uinfy it with the kepler disposition

In [214]:
print(TOI_data_selected["target"].unique())


['FP' 'PC' 'KP' 'APC' 'FA' 'CP']


In [215]:
tess_mapping = {
    "CP": "CONFIRMED",      # Confirmed Planet
    "PC": "CANDIDATE",      # Planet Candidate
    "FP": "FALSE POSITIVE", # False Positive
    "KP": "CANDIDATE",     # Kepler-like Candidate
    "FA":"FALSE POSITIVE",  #flase alarm
    "APC":"CANDIDATE"       #ambiguous planetery candidate

}

TOI_data_selected["target"] = TOI_data_selected["target"].map(tess_mapping)

C:\Users\Mohamed Adel\AppData\Local\Temp\ipykernel_31824\1100168568.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  TOI_data_selected["target"] = TOI_data_selected["target"].map(tess_mapping)


now the data is ready for merging


In [217]:
# First, select only the columns we want to keep (features + target)
common_columns = [
    "orbital_period", "transit_duration", "transit_depth", "planet_radius",
    "insolation_flux", "equilibrium_temperature", "stellar_teff",
    "stellar_logg", "stellar_radius", "stellar_mag", "ra", "dec",
    "target"
]

KOI_final = KOI_data_selected[common_columns].copy()
TOI_final = TOI_data_selected[common_columns].copy()

# Concatenate the datasets
combined_data = pd.concat([KOI_final, TOI_final], ignore_index=True)

# Optional: shuffle the data
combined_data = combined_data.sample(frac=1, random_state=42).reset_index(drop=True)

# Check the merged dataset
combined_data.shape
combined_data.head()


,orbital_period,transit_duration,transit_depth,planet_radius,insolation_flux,equilibrium_temperature,stellar_teff,stellar_logg,stellar_radius,stellar_mag,ra,dec,target
0,5.541111,2.4670,633.0,2.04482,258.843,1117.0,5332.0,4.56429,0.82000,9.2789,99.238421,-58.016153,CANDIDATE
1,6.685438,5.0900,89.2,1.25000,276.020,1040.0,5808.0,4.32600,1.15900,15.1350,295.553740,46.029110,CANDIDATE
2,25.585467,2.2800,421.1,2.39000,141.600,878.0,5767.0,4.43800,1.00000,15.1610,289.118680,43.077389,FALSE POSITIVE
3,0.584148,1.6192,66.2,0.87000,4882.900,2130.0,5427.0,4.34700,1.03100,13.9490,292.978670,42.056492,FALSE POSITIVE
4,2.539558,3.0820,24200.0,10.50980,199.853,1047.0,3931.6,4.33000,1.23369,13.0427,184.044295,-66.055079,CANDIDATE


DAta is great till now 

In [218]:
combined_data.isnull().sum()

orbital_period             0
transit_duration           0
transit_depth              0
planet_radius              0
insolation_flux            0
equilibrium_temperature    0
stellar_teff               0
stellar_logg               0
stellar_radius             0
stellar_mag                0
ra                         0
dec                        0
target                     0
dtype: int64

In [222]:
#save the data to work on it in the model
combined_data.to_csv("../datasets/combined_data.csv")

print("DATA SAVED !!")

with open("../datasets/combined_features.json", "w") as f:
    json.dump({"features": selected_merge_features}, f)

print("✅ Features and target saved")


DATA SAVED !!
✅ Features and target saved
